# 01 DeepGO API 预测

调用 [DeepGOWeb](https://deepgo.cbrc.kaust.edu.sa/) REST API（`/deepgo/api/create`）对 FASTA 文件批量预测 GO 功能注释。

- 单次请求最多 10 条序列，按批提交、逐批落盘（防止中断丢失）。
- 参数：`data_format=fasta`、`data=FASTA文本`、`threshold`（模型输出阈值）、`version=1.0.28`。
- 本仓库所有 DeepGO 预测均使用 `threshold=0.1`（模型输出的置信度阈值，不是下游判定阈值）。


In [ ]:
# -*- coding: utf-8 -*-
import json, os, time, argparse, requests

API_URL = "https://deepgo.cbrc.kaust.edu.sa/deepgo/api/create"
BATCH_SIZE = 10          # API 单次最多 10 条序列
VERSION = "1.0.28"       # API 版本字段
THRESHOLD = 0.1          # 预测置信度阈值（本仓库最终口径）
RETRIES = 3
TIMEOUT = 180

def parse_fasta(fasta_path):
    """解析 FASTA，返回 [(header, sequence), ...]"""
    records, header, seq_parts = [], None, []
    with open(fasta_path, "r", encoding="utf-8", errors="ignore") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if header is not None:
                    records.append((header, "".join(seq_parts)))
                header, seq_parts = line[1:].strip(), []
            else:
                seq_parts.append(line)
    if header is not None:
        records.append((header, "".join(seq_parts)))
    return records

def submit_batch(batch_records, threshold, log_path=None):
    """提交一批，返回 JSON；失败返回 None（带重试）。"""
    fasta_block = "\n".join(">" + h + "\n" + s for h, s in batch_records)
    payload = {"data_format": "fasta", "data": fasta_block,
               "threshold": str(threshold), "version": VERSION}
    for attempt in range(1, RETRIES + 1):
        try:
            resp = requests.post(API_URL, data=payload, timeout=TIMEOUT)
            if 200 <= resp.status_code < 300:
                return resp.json()
            if log_path:
                with open(log_path, "a", encoding="utf-8") as fh:
                    fh.write(f"[attempt {attempt}] HTTP {resp.status_code}: {resp.text[:200]}\n")
        except requests.exceptions.RequestException as exc:
            if log_path:
                with open(log_path, "a", encoding="utf-8") as fh:
                    fh.write(f"[attempt {attempt}] RequestException: {exc}\n")
        time.sleep(5 * attempt)
    return None

def process_fasta(fasta_path, out_dir, threshold=THRESHOLD):
    """处理单个 FASTA：分批提交、合并保存为 <base>.json，并写 <base>.log"""
    base = os.path.splitext(os.path.basename(fasta_path))[0]
    out_path = os.path.join(out_dir, base + ".json")
    log_path = os.path.join(out_dir, base + ".log")
    records = parse_fasta(fasta_path)
    results, total = [], (len(records) + BATCH_SIZE - 1) // BATCH_SIZE
    for i in range(0, len(records), BATCH_SIZE):
        batch = records[i:i + BATCH_SIZE]
        json_res = submit_batch(batch, threshold, log_path)
        results.append({"batch": i // BATCH_SIZE + 1,
                        "headers": [h for h, _ in batch],
                        "data": json_res})
        with open(out_path, "w", encoding="utf-8") as fh:   # 逐批落盘
            json.dump(results, fh, ensure_ascii=False, indent=2)
        time.sleep(2)
    return results

# 示例用法（路径按需修改）：
# process_fasta("../data/fasta/2026_predict/dataset_easy_redundant.fasta", "out_dir")


### 使用说明
- 修改最后一行路径后运行即可。输出 `<basename>.json` 的结构：
  `[{batch, headers, data: {predictions: [{protein_info, functions: [{functions: [[GO, term, score], ...]}]}]}}]`
- 本仓库未内置大体积预测 JSON（GitHub 体积考虑），可用本脚本重新生成。
- NetGO3/NetGO4 预测在 [NetGO 服务器](https://dmiip.sjtu.edu.cn/NetGO/) 上运行，结果为 5 列制表符分隔 txt：`entry\tGO\tscore\tcat_type\tterm`。
